In [ ]:
#importing the needed libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns
from tqdm import tqdm


from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:


# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")


In [ ]:

df_food.head() # to read the first few rows

In [ ]:

df_food.info()

In [ ]:

df_food.describe() #to show the statistical info of the data

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df_food, "Delivery_Time")


In [ ]:
# Task 1: Drop the "Order_ID" column from the data

df_food = df_food.drop('Order_ID', axis=1)
#X = df.drop("Sales", axis=1).astype(float)
#y = df['Sales'].astype(float)

In [ ]:
# Handle missing values

missing_values = df_food.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])

#Fill missing values with mean for each column
df_food = df_food.fillna(df_food.mean())

# Analyze missing values
#missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
#missing_data = pd.DataFrame({
    #'Column': missing_percentage.index,
    #'Missing_Percentage': missing_percentage.values
#})
#missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

#print("Missing Data Analysis:")
#missing_data.head(10)

def check_missing_values(df_food):
  missing_values = df_food.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_food)




In [ ]:
# check and remove duplicates

def check_duplicates(df_food):
  duplicates = df_food.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_food.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)


In [ ]:
# Encode categorical columns - which converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_food[col] = le.fit_transform(df_food[col].astype(str))

df_food.head()


#def one_hot_encode(y, num_classes):
    #y = np.array(y)
    #m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    #one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    #for i in range(m):
        # Identify which class this sample belongs to
        #class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        #one_hot[i, class_label] = 1

    #return one_hot

In [ ]:
# Apply StandardScalar

from sklearn.preprocessing import StandardScaler

numerical_cols = df_food.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])
df_food.head()


In [ ]:
# check for imbalance

def check_target_imbalance(df_food, target_column):
  print("Target Distribution:")
  print(df_food[target_column].value_counts(normalize=True))
  sns.countplot(x=df_food[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_food, "Delivery_Time")

In [ ]:
# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)

  def gradient_descent(X, y, learning_rate, n_iters=500):
    m, n = X.shape  # m rows, n columns (dimensions)
    theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
    losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

n_splits = 5

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for linear regression results for each fold
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)





In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


import numpy as np

# Sample data
X = np.random.rand(100, 5)
y = np.random.randint(0, 2, 100)

# KFold Cross-Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor()

# Get scores for each fold
scores = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')
print("KFold scores:", scores)
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# StratifiedKFold for the imbalanced data
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(model, X, y, cv=skfold, scoring='accuracy')
print("\nStratifiedKFold scores:", scores_strat)
print(f"Mean accuracy: {scores_strat.mean():.3f}")


In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
print("Random Forest R²:", rf.score(X_test, y_test))

# Define features (X) and target (y)
feature_cols = ['']
X = df_food[feature_cols]
y = df_food['Delievry_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")







In [ ]:

# Plot feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 6))
plt.hist(valid_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Delievery Time')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here:



